In [2]:
import pickle

with open("chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

In [3]:
from langchain_core.documents import Document

docs = [Document(page_content=text) for text in all_chunks]

In [4]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\JOHN\anaconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vdb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="route"
)

c:\Users\JOHN\anaconda3\envs\nlp\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [6]:
retriever = vdb.as_retriever(search_kwargs={"k":10})

In [7]:
query = "what is ML"

In [8]:
results = vdb.similarity_search_with_score(query, k=3)

In [9]:
context = "\n\n".join(
    [doc.page_content for doc, score in results]
)

In [10]:
context

"ML\n\nIn basic terms, ML is the process of training a piece of software, called a model , to make useful predictions or generate content\n(like text, images, audio, or video) from\ndata.\n\nSome view ML as a universal tool that can be applied to all problems. In\nreality, ML is a specialized tool suitable only for particular problems. You\ndon't want to implement a complex ML solution when a simpler non-ML solution\nwill work."

In [11]:
prompt = f"""
You are an AI assistant in a Retrieval-Augmented Generation (RAG) system.

You must answer the user's question ONLY using the information provided in the retrieved context.

Instructions:
1. Use ONLY the retrieved context below.
2. Do NOT use your own knowledge.
3. If the answer is not contained in the context, reply exactly:
   "I cannot find the answer in the provided context."
4. Do not make up information or assumptions.
5. If multiple retrieved chunks contain relevant information, combine them into one coherent answer.
6. Keep the answer clear and concise.

======================
Retrieved Context:
{context}
======================

User Question:
{query}

Answer:
"""

In [12]:
import google.generativeai as genai

genai.configure(api_key="your api key")

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(prompt)

print(response.text)

C:\Users\JOHN\AppData\Local\Temp\ipykernel_16724\3862538532.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]